In [8]:
import pandas as pd

movies_df = pd.read_csv('/Users/alexgarcia/Desktop/CityUniversity/MSDS/DS522 Data Aquisition and Analytics/Modules/Module5/PE05/movies.csv')
ratings_df = pd.read_excel('/Users/alexgarcia/Desktop/CityUniversity/MSDS/DS522 Data Aquisition and Analytics/Modules/Module5/PE05/ratings.xlsx')

print(movies_df.head())
print(ratings_df.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [9]:
df = pd.merge(movies_df, ratings_df, on='movieId')

print(df.head())
print(df.shape)

   movieId             title                                       genres  \
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
1        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
2        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
3        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
4        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   

   userId  rating   timestamp  
0       1     4.0   964982703  
1       5     4.0   847434962  
2       7     4.5  1106635946  
3      15     2.5  1510577970  
4      17     4.5  1305696483  
(100836, 6)


In [10]:
# Remove rows with missing values
df.dropna(inplace=True)

# Convert timestamp to readable datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Extract release year from title (e.g. "Toy Story (1995)")
df['release_year'] = df['title'].str.extract(r'\((\d{4})\)')
df.dropna(subset=['release_year'], inplace=True)
df['release_year'] = df['release_year'].astype(int)

print(df.head())
print(df.dtypes)

   movieId             title                                       genres  \
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
1        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
2        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
3        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
4        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   

   userId  rating           timestamp  release_year  
0       1     4.0 2000-07-30 18:45:03          1995  
1       5     4.0 1996-11-08 06:36:02          1995  
2       7     4.5 2005-01-25 06:52:26          1995  
3      15     2.5 2017-11-13 12:59:30          1995  
4      17     4.5 2011-05-18 05:28:03          1995  
movieId                 int64
title                     str
genres                    str
userId                  int64
rating                float64
timestamp       datetime64[s]
release_year            int64
dty

In [11]:
movie_stats = df.groupby('title').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

top_movie = (movie_stats[movie_stats['rating_count'] >= 100]
             .sort_values('avg_rating', ascending=False)
             .iloc[0])

print(f"Highest Rated Movie: {top_movie['title']}")
print(f"Average Rating: {top_movie['avg_rating']:.2f}")

Highest Rated Movie: Shawshank Redemption, The (1994)
Average Rating: 4.43


In [12]:
genre_df = df.copy()
genre_df['genres'] = genre_df['genres'].str.split('|')
genre_df = genre_df.explode('genres')

top_genre = (genre_df.groupby('genres')
             .agg(avg_rating=('rating', 'mean'))
             .sort_values('avg_rating', ascending=False)
             .reset_index())

print(top_genre.head(10))

        genres  avg_rating
0    Film-Noir    3.920115
1          War    3.808294
2  Documentary    3.797785
3        Crime    3.658294
4        Drama    3.656168
5      Mystery    3.632460
6    Animation    3.629937
7         IMAX    3.618335
8      Western    3.583938
9      Musical    3.563678


In [13]:
yearly_ratings = (df.groupby('release_year')
                  .agg(avg_rating=('rating', 'mean'))
                  .reset_index()
                  .sort_values('release_year'))

print(yearly_ratings)

     release_year  avg_rating
0            1902    3.500000
1            1903    2.500000
2            1908    4.000000
3            1915    2.000000
4            1916    3.600000
..            ...         ...
101          2014    3.512879
102          2015    3.410386
103          2016    3.387261
104          2017    3.578091
105          2018    3.483516

[106 rows x 2 columns]


In [14]:
df['era'] = df['release_year'].apply(lambda x: 'Old' if x < 2000 else 'New')

print(df[['title', 'release_year', 'era']].drop_duplicates().head(10))

                                  title  release_year  era
0                      Toy Story (1995)          1995  Old
215                      Jumanji (1995)          1995  Old
325             Grumpier Old Men (1995)          1995  Old
377            Waiting to Exhale (1995)          1995  Old
384  Father of the Bride Part II (1995)          1995  Old
433                         Heat (1995)          1995  Old
535                      Sabrina (1995)          1995  Old
589                 Tom and Huck (1995)          1995  Old
597                 Sudden Death (1995)          1995  Old
613                    GoldenEye (1995)          1995  Old


In [15]:
genre_label_map = {
    'Comedy': 'Popular',
    'Drama':  'Classic',
    'Action': 'Exciting'
}

df['primary_genre'] = df['genres'].str.split('|').str[0]
df['genre_label'] = df['primary_genre'].map(genre_label_map).fillna('Other')

print(df[['title', 'primary_genre', 'genre_label']].drop_duplicates().head(10))

                                  title primary_genre genre_label
0                      Toy Story (1995)     Adventure       Other
215                      Jumanji (1995)     Adventure       Other
325             Grumpier Old Men (1995)        Comedy     Popular
377            Waiting to Exhale (1995)        Comedy     Popular
384  Father of the Bride Part II (1995)        Comedy     Popular
433                         Heat (1995)        Action    Exciting
535                      Sabrina (1995)        Comedy     Popular
589                 Tom and Huck (1995)     Adventure       Other
597                 Sudden Death (1995)        Action    Exciting
613                    GoldenEye (1995)        Action    Exciting
